In [1]:
import pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, PercentFormatter
from datetime import datetime, timezone

# import binance files
from binance.binance_funding import binance_funding, binance_basis
from binance.binance_candles import binance_candles
from binance.binance_oi import binance_oi
from binance.binance_liquidation import binance_liquidation
from binance.binance_long_short import binance_global_account_long_short, binance_top_account_long_short, binance_top_position_long_short

# import hyperliquid files
from hyperliquid.hl_candles import hl_candles
from hyperliquid.hl_funding import hl_funding
from hyperliquid.hl_oi import hl_oi
from hyperliquid.hl_liquidation import hl_liquidation
from hyperliquid.hl_price_impact import hl_price_impact
from hyperliquid.hl_vault import hl_vault_pnl

In [3]:
# token info
token_name_binance = "BTCUSDT" # Binance API token name
token_name_hl = "BTC" # HL API token name
cg_token_name_binance = "BTCUSD_PERP" # CoinGlass API Binance token name
cg_token_name_hl = "BTC-USD" # CoinGlass API HL token name
contract_type = "PERPETUAL"
interval = "4h"
start_time = datetime(2026, 1, 15, 0, 0, 0, tzinfo=timezone.utc) # start date: jan 15th
end_time = datetime(2026, 2, 10, 0, 0, 0, tzinfo=timezone.utc) # end date: feb 10th
limit = 1000

# convert time to unix format
start_time_stamp = int(start_time.timestamp() * 1000)
end_time_stamp = int(end_time.timestamp() * 1000)

In [4]:
# hyperliquid dataframes
hl_candles_df = hl_candles(token_name_hl, start_time_stamp, end_time_stamp, interval) # candles
hl_funding_df = hl_funding(token_name_hl, start_time_stamp, end_time_stamp) # funding rate
hl_oi_df = hl_oi(cg_token_name_hl, interval, start_time_stamp, end_time_stamp) # open interest
hl_liquidation_df = hl_liquidation(cg_token_name_hl, interval, start_time_stamp, end_time_stamp) # liquidation

In [ ]:
# binance dataframes
binance_candles_df = binance_candles(token_name_binance, interval, start_time_stamp, end_time_stamp, limit) # candles
binance_funding_df = binance_funding(token_name_binance, start_time_stamp, end_time_stamp, limit) # funding rate
binance_basis_df = binance_basis(token_name_binance, contract_type, interval, limit, start_time_stamp, end_time_stamp) # basis

In [5]:
binance_global_account_long_short_df = binance_global_account_long_short(cg_token_name_binance, interval, start_time_stamp, end_time_stamp) # global account long/short
binance_top_account_long_short_df = binance_top_account_long_short(cg_token_name_binance, interval, start_time_stamp, end_time_stamp) # top account long/short
binance_top_position_long_short_df = binance_top_position_long_short(cg_token_name_binance, interval, start_time_stamp, end_time_stamp) # top position long/short
binance_liquidation_df = binance_liquidation(cg_token_name_binance, interval, start_time_stamp, end_time_stamp) # liquidation
binance_oi_df = binance_oi(cg_token_name_binance, interval, start_time_stamp, end_time_stamp) # open interest

In [ ]:
# compute hyperliquid typical price and notional volume
hl_candles_df["typical_price"] = (hl_candles_df["high"] + hl_candles_df["low"] + hl_candles_df["close"]) / 3
hl_candles_df["notional_volume"] = hl_candles_df["typical_price"] * hl_candles_df["volume"]

In [ ]:
# compute hyperliquid volume multiplier
def hl_volume_multiplier(hl_candles_df: pd.DataFrame):

    # filter df between jan. 15th and jan. 29th
    df_jan15_jan29 = hl_candles_df[
        (hl_candles_df["start_time"] >= pd.Timestamp("2026-01-15")) &
        (hl_candles_df["start_time"] <= pd.Timestamp("2026-01-29"))
    ]

    # filter df between jan. 29th and feb. 07th
    df_jan29_feb07 = hl_candles_df[
        (hl_candles_df["start_time"] >= pd.Timestamp("2026-01-29")) &
        (hl_candles_df["start_time"] <= pd.Timestamp("2026-02-07"))
    ]

    # compute volume multiplier
    avg_vlm_jan15_jan29 = df_jan15_jan29["notional_volume"].mean()
    avg_vlm_jan29_feb07 = df_jan29_feb07["notional_volume"].mean()
    volume_multiplier = avg_vlm_jan29_feb07 / avg_vlm_jan15_jan29
    return volume_multiplier

print(hl_volume_multiplier(hl_candles_df))

In [ ]:
# plot price and notional volume
def plot_price_volume(hl_candles_df: pd.DataFrame, binance_candles_df: pd.DataFrame):

    fig, ax1 = plt.subplots(figsize = (14, 7))
    ax2 = ax1.twinx()

    ax1.plot(hl_candles_df["start_time"], hl_candles_df["typical_price"], color = "#A900B8", label = "Price")
    ax2.plot(hl_candles_df["start_time"], hl_candles_df["notional_volume"], color = "#00B8A9", label = "Hyperliquid")
    ax2.plot(binance_candles_df["start_time"], binance_candles_df["notional_volume"], color = "#F0B90A", label = "Binance")
    ax1.set_xlabel("Time")
    ax1.set_ylabel("Price (USD)")
    ax2.set_ylabel("Notional Volume (USD)")

    # formatting
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x/1e9:.1f}B"))

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
    handles1 + handles2,
    labels1 + labels2
    )
    plt.title("BTC Perps Notional Volume vs. Time")

plot_price_volume(hl_candles_df, binance_candles_df)

In [ ]:
# transform funding into an 8h interval dataframe to match binance data
def hl_funding_8h(hl_funding_df: pd.DataFrame):

    hl_funding_8h_df = (
        hl_funding_df
        .set_index("time")
        .resample("8h")["fundingRate"]
        .sum()
        .reset_index(name="fundingRate")
    )

    return hl_funding_8h_df

hl_funding_8h_df = hl_funding_8h(hl_funding_df)

In [ ]:
# plot funding (note: 8h intervals)
def plot_funding(hl_funding_8h_df: pd.DataFrame, binance_funding_df: pd.DataFrame):

    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(binance_funding_df["fundingTime"], binance_funding_df["fundingRate"], color = "#F0B90A", label = "Binance")
    ax.plot(hl_funding_8h_df["time"], hl_funding_8h_df["fundingRate"], color = "#00B8A9", label = "Hyperliquid")
    ax.set_xlabel("Time")
    ax.set_ylabel("Funding Rate")

    # formatting
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda y, _: f"{y*100:.3f}%")
    )
    ax.legend()
    plt.title("BTC Perps: Funding")

plot_funding(hl_funding_8h_df, binance_funding_df)

In [ ]:
# compute exact basis rate for binance
binance_basis_df["basisRate_exact"] = (binance_basis_df["futuresPrice"] - binance_basis_df["indexPrice"]) / binance_basis_df["futuresPrice"]

In [ ]:
# plot basis
def plot_basis(hl_funding_df: pd.DataFrame):

    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(hl_funding_df["time"], hl_funding_df["premium"], color = "#00B8A9", label = "Hyperliquid")
    ax.plot(binance_basis_df["timestamp"], binance_basis_df["basisRate_exact"], color = "#F0B90A", label = "Binance")
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis")

    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    plt.title("BTC Perps: Basis")

plot_basis(hl_funding_df)

In [ ]:
# plot open interest
def plot_oi(hl_oi_df: pd.DataFrame, binance_oi_df: pd.DataFrame):
    
    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(hl_oi_df["time"], hl_oi_df["close"], color = "#00B8A9", label = "Hyperliquid")
    ax.plot(binance_oi_df["time"], binance_oi_df["close"], color = "#F0B90A", label = "Binance")
    ax.set_xlabel("Time")
    ax.set_ylabel("Open Interest (USD)")

    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1e9:.1f}B"))
    fig.autofmt_xdate()
    plt.title("ETH Perps: Open Interest")

plot_oi(hl_oi_df, binance_oi_df)

In [ ]:
# compute contract turnover (volume/OI)
binance_turnover_df = pd.DataFrame({"end_time": binance_candles_df["end_time"], "turnover": binance_candles_df["notional_volume"] / binance_oi_df["close"]})
binance_turnover_df["rolling_turnover"] = binance_turnover_df["turnover"].rolling(6).mean() # add rolling turnover

hl_turnover_df = pd.DataFrame({"end_time": hl_candles_df["end_time"], "turnover": hl_candles_df["notional_volume"] / hl_oi_df["close"]})
hl_turnover_df["rolling_turnover"] = hl_turnover_df["turnover"].rolling(6).mean() # add rolling tunrover

In [ ]:
# plot turnover (volume/OI)
def plot_turnover(hl_turnover_df: pd.DataFrame, binance_turnover_df: pd.DataFrame):
    
    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(hl_turnover_df["end_time"], hl_turnover_df["rolling_turnover"], color = "#00B8A9", label = "Hyperliquid")
    ax.plot(binance_turnover_df["end_time"], binance_turnover_df["rolling_turnover"], color = "#F0B90A", label = "Binance")
    ax.set_xlabel("Time")
    ax.set_ylabel("Multiple over Open Interest")

    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:.0f}x"))
    fig.autofmt_xdate()
    plt.title("BTC Perps: Rolling Volume-to-Open Interest Ratio")


plot_turnover(hl_turnover_df, binance_turnover_df)

In [ ]:
# compute liquidation intensity
hl_liquidation_df["intensity"] = hl_liquidation_df["long_liquidation_usd"] / hl_oi_df["close"]
binance_liquidation_df["intensity"] = binance_liquidation_df["long_liquidation_usd"] / binance_oi_df["close"]

In [ ]:
# plot liquidation intensity
def plot_liquidation(hl_liquidation_df: pd.DataFrame, binance_liquidation_df: pd.DataFrame):
    
    fig, ax = plt.subplots(figsize = (14, 7))
    ax.bar(hl_liquidation_df["time"], hl_liquidation_df["intensity"], color = "#00B8A9", label = "Hyperliquid")
    ax.bar(binance_liquidation_df["time"], binance_liquidation_df["intensity"], color = "#F0B90A", label = "Binance")
    ax.set_xlabel("Time")
    ax.set_ylabel("Percentage of Open Interest")

    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y * 100:.0f}%"))
    fig.autofmt_xdate()
    plt.title("BTC Perps: Long Liquidation Intensity")

plot_liquidation(hl_liquidation_df, binance_liquidation_df)

In [ ]:
# create Hyperliquid price impact dataframe and compute bid impact bps
query_id = 7920398
hl_price_impact_df = hl_price_impact(query_id)
hl_price_impact_df["bid_impact_bps"] = (hl_price_impact_df["mid_px"] - hl_price_impact_df["impact_bid_px"]) / hl_price_impact_df["mid_px"] * 10000
hl_price_impact_df.head()

In [ ]:
def plot_hl_bid_impact(hl_price_impact_df: pd.DataFrame):

    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(hl_price_impact_df["time"], hl_price_impact_df["bid_impact_bps"], color = "#00B8A9")
    ax.set_title("BTC Perps: Hyperliquid Bid Impact Deviation")
    ax.set_xlabel("Time")
    ax.set_ylabel("Distance to Mid Price (bps)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    plt.show()

plot_hl_bid_impact(hl_price_impact_df)

In [ ]:
# plot binance global account long-short ratio
def plot_binance_long_short(binance_global_account_long_short_df: pd.DataFrame):

    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(binance_global_account_long_short_df["time"], binance_global_account_long_short_df["global_account_long_short_ratio"], color = "#F0B90A")
    ax.set_title("BTC Perps: Binance Global Account Long-Short Ratio")
    ax.set_xlabel("Time")
    ax.set_ylabel("Ratio")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    plt.show()

plot_binance_long_short(binance_global_account_long_short_df)

In [ ]:
# vault addresses
hlp_liquidator = "0xdfc24b077bc1425ad1dea75bcb6f8158e10df303"
period = "allTime"

# filter the dataframe between september 2025 and march 2026
hl_vault_pnl_df = hl_vault_pnl(hlp_liquidator, period)
sept_to_mar_pnl = hl_vault_pnl_df[
    hl_vault_pnl_df["timestamp"].between("2025-09-01", "2026-03-1")
]

sept_to_mar_pnl

In [ ]:
# plot hlp cumulative pnl from september 2025 to march 2026
def plot_vault_pnl(sept_to_mar_pnl):

    fig, ax = plt.subplots(figsize = (14, 7))
    ax.plot(sept_to_mar_pnl["timestamp"], sept_to_mar_pnl["pnl"], color = "#00B8A9")
    ax.set_xlabel("Time")
    ax.set_ylabel("PnL")

    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1e6:.1f}M"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate()
    plt.title("HLP Cumulative PnL (Sept 2025 - Mar 2026)")

plot_vault_pnl(sept_to_mar_pnl)